# 🏭 Petrochemical Product Yield Prediction

## About This Notebook

In this notebook, we will predict the **Product Yield (Tons)** from a petrochemical plant using sensor and process data.

**What we'll do step by step:**
1. Load and explore the dataset
2. Clean and prepare the data
3. Visualize important patterns
4. Train multiple ML models
5. Compare models and pick the best one
6. Understand which features matter most

> 🟢 **Beginner Tip:** Each section has simple explanations. Take it one step at a time!

## 📦 Step 1: Import Libraries

Think of libraries as toolboxes. We import them at the start so we can use their tools throughout the notebook.

In [ ]:
# Data handling
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ All libraries loaded!')

## 📂 Step 2: Load the Dataset

We load the CSV file into a **DataFrame** — think of it like an Excel sheet inside Python.

In [ ]:
df = pd.read_csv('/kaggle/input/petrochemical-advanced-data/petrochemical_advanced_data.csv')

print(f'📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

Before building models, we **explore** the data to understand it better.

### 3.1 Basic Info

In [ ]:
print('=== Data Types & Non-Null Counts ===')
print(df.info())

print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
print('=== Statistical Summary ===')
df.describe().T

### 3.2 Target Variable Distribution

Our goal is to predict `Product_Yield_Tons`. Let's see how it's distributed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['Product_Yield_Tons'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Product Yield (Tons)', fontsize=13)
axes[0].set_xlabel('Product Yield (Tons)')
axes[0].set_ylabel('Count')

# Box plot
axes[1].boxplot(df['Product_Yield_Tons'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Box Plot of Product Yield (Tons)', fontsize=13)
axes[1].set_ylabel('Product Yield (Tons)')

plt.tight_layout()
plt.show()

print(f'Mean Yield: {df["Product_Yield_Tons"].mean():.2f} Tons')
print(f'Std  Yield: {df["Product_Yield_Tons"].std():.2f} Tons')

### 3.3 Categorical Column Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['Unit_Name', 'Catalyst_Type']):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
    ax.set_title(f'{col} Distribution', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap

Correlation tells us how strongly two variables are related. Values close to **+1 or -1** mean strong relationships.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(12, 8))
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

### 3.5 Top Feature Relationships with Target

In [ ]:
top_features = ['Sensor_Health_Index', 'Feedstock_Flow_m3h',
                'Reactor_Temp_C', 'Reactor_Pressure_Bar']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['Product_Yield_Tons'],
                    alpha=0.3, color='steelblue', s=5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Product Yield (Tons)')
    axes[i].set_title(f'{feat} vs Yield')

plt.suptitle('Feature vs Product Yield Scatter Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 🛠️ Step 4: Feature Engineering & Preprocessing

Machine learning models only understand numbers. We convert text columns to numbers and create new useful features from the timestamp.

In [ ]:
# Parse timestamp and extract time features
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour']  = df['Timestamp'].dt.hour    # 0-23 — time of day
df['Month'] = df['Timestamp'].dt.month   # 1-12 — seasonality

# Encode categorical columns (text → numbers)
le = LabelEncoder()
for col in ['Unit_Name', 'Catalyst_Type']:
    df[col] = le.fit_transform(df[col])
    print(f'Encoded {col}')

print('\n✅ Feature engineering complete!')

## ✂️ Step 5: Train-Test Split

We split data into:
- **Training set (80%)** — model learns from this
- **Test set (20%)** — we test on unseen data to check real performance

> 🟢 **Note:** We drop `Energy_Intensity` because it's derived from the target — using it would cause **data leakage**.

In [ ]:
# Define features (X) and target (y)
drop_cols = ['Timestamp', 'Product_Yield_Tons', 'Energy_Intensity']
features = [c for c in df.columns if c not in drop_cols]

X = df[features]
y = df['Product_Yield_Tons']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Number of features: {X_train.shape[1]}')

## 🤖 Step 6: Train Machine Learning Models

We train 3 different models and compare them:

| Model | Type | Good For |
|---|---|---|
| Linear Regression | Simple baseline | When relationships are linear |
| Random Forest | Ensemble of trees | Complex non-linear patterns |
| Gradient Boosting | Boosted trees | High accuracy, state-of-the-art |

In [ ]:
models = {
    'Linear Regression'  : LinearRegression(),
    'Random Forest'      : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)          # Train
    preds = model.predict(X_test)        # Predict on test set

    r2  = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    results[name] = {'R² Score': r2, 'MAE': mae, 'RMSE': rmse, 'predictions': preds}
    print(f'{name:22s} → R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}')

## 📊 Step 7: Compare Models

- **R² Score** — closer to 1.0 is better (explains variance)
- **MAE** — lower is better (average prediction error in tons)
- **RMSE** — lower is better (penalizes large errors more)

In [ ]:
results_df = pd.DataFrame({
    name: {'R² Score': v['R² Score'], 'MAE': v['MAE'], 'RMSE': v['RMSE']}
    for name, v in results.items()
}).T

print('=== Model Comparison ===')
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['R² Score', 'MAE', 'RMSE']
colors  = ['steelblue', 'coral', 'mediumseagreen']

for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].astype(float)
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor='white')
    ax.set_title(metric, fontsize=13)
    ax.set_xticklabels(vals.index, rotation=15, ha='right')
    # Label each bar
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 🏆 Step 8: Best Model — Random Forest Deep Dive

In [ ]:
# Actual vs Predicted plot
best_preds = results['Random Forest']['predictions']

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.3, color='steelblue', s=10, label='Predictions')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Yield (Tons)')
plt.ylabel('Predicted Yield (Tons)')
plt.title('Random Forest: Actual vs Predicted Yield', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test.values - best_preds

plt.figure(figsize=(8, 4))
plt.scatter(best_preds, residuals, alpha=0.3, color='coral', s=10)
plt.axhline(0, color='black', linewidth=1.5, linestyle='--')
plt.xlabel('Predicted Yield')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

## 🔑 Step 9: Feature Importance

Which plant variables drive the yield prediction the most?

In [ ]:
rf_model = models['Random Forest']
importance = pd.Series(rf_model.feature_importances_, index=features)\
               .sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors_fi = ['steelblue' if v < importance.max() else 'gold' for v in importance.values]
importance.plot(kind='barh', color=colors_fi, edgecolor='white')
plt.xlabel('Feature Importance Score')
plt.title('Random Forest — Feature Importance', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 3 Most Important Features:')
for feat, score in importance.sort_values(ascending=False).head(3).items():
    print(f'  {feat:30s}: {score:.4f}')

## ✅ Step 10: Conclusion

### What We Learned

**Dataset:** 10,000 hourly plant readings from 3 units with 16 features tracking reactor conditions, energy usage, and sensor health.

**Best Model: Random Forest Regressor**

| Metric | Score |
|---|---|
| R² Score | **0.9992** |
| MAE | **0.21 Tons** |
| RMSE | **0.40 Tons** |

The model explains **99.92%** of the variance in product yield — a near-perfect fit.

**Key Insights:**
- 🥇 `Sensor_Health_Index` is by far the strongest predictor of yield (~61% importance)
- 🥈 `Feedstock_Flow_m3h` is the second most important (~39% importance)
- Both together explain virtually all of the predictive signal
- `Energy_Intensity` was deliberately excluded to prevent data leakage

**Practical Recommendation:**
> Maintaining **sensor health** and controlling **feedstock flow rates** are the two levers with the highest impact on product yield in this petrochemical plant. Plant operators should prioritize sensor calibration and stable feedstock delivery.

**Next Steps:**
- Try XGBoost or LightGBM for potential further gains
- Deploy the model as a real-time yield monitoring dashboard
- Investigate anomalies where yield deviates from prediction